In [1]:
import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows() 

DataTransformerRegistry.enable('default')

In [2]:
url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv'
buildings = pd.read_csv(url)
buildings.head()

,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
0,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,1975,1975,144,1,1,0,Unusual,Unusual,Not provided
1,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
2,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
3,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
4,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided


In [5]:
buildings.loc[buildings['Year Constructed'] == 0, 'Year Constructed'] = np.nan
buildings.loc[buildings['Square Footage'] == 0, 'Square Footage'] = np.nan

agency_sqft = buildings.groupby('Agency Name')['Square Footage'].sum().reset_index()
agency_sqft = agency_sqft.nlargest(10, 'Square Footage')
agency_sqft

,Agency Name,Square Footage
33,University of Illinois,25018006.0
7,Department of Corrections,15120750.0
32,Southern Illinois University,8709473.0
8,Department of Human Services,8466774.0
15,Department of Transportation,5659737.0
10,Department of Military Affairs,4579470.0
6,Department of Central Management Services,4260911.0
11,Department of Natural Resources,3937319.0
29,Northern Illinois University,3751095.0
27,Illinois State University,2960272.0


In [6]:
chart1 = alt.Chart(agency_sqft).mark_bar().encode(
    alt.X('Square Footage:Q', title='Total square footage'),
    alt.Y('Agency Name:N', sort='-x', title=None),
    tooltip=['Agency Name', alt.Tooltip('Square Footage:Q', format=',')]
).properties(
    title='Which agencies have the most building space?'
)
chart1

alt.Chart(...)

In [7]:
built = buildings.dropna(subset=['Year Constructed']).copy()
built['Decade'] = (built['Year Constructed'] // 10 * 10).astype(int)

usage_decade = built.groupby(['Usage Description', 'Decade']).size().reset_index(name='Buildings')
usage_decade.head()

,Usage Description,Decade,Buildings
0,Assembly,1750,6
1,Assembly,1800,2
2,Assembly,1810,4
3,Assembly,1830,4
4,Assembly,1840,11


In [9]:
selection = alt.selection_point(fields=['Usage Description']) 
color = alt.condition(selection, alt.value('#417548'), alt.value('lightgray'))

usage_bars = alt.Chart(usage_decade).mark_bar().encode(
    alt.X('sum(Buildings):Q', title='Number of buildings'), 
    alt.Y('Usage Description:N', sort='-x', title=None), 
    color=color, 
    tooltip=['Usage Description', alt.Tooltip('sum(Buildings):Q', title='Buildings')]
).add_params(
    selection
).properties(
    title='Select(click) building type', width=250
)

decade_bars = alt.Chart(usage_decade).mark_bar().encode(
    alt.X('Decade:O', title='Decade built'), 
    alt.Y('sum(Buildings):Q', title='Number of buildings'),
    tooltip=['Decade', alt.Tooltip('sum(Buildings):Q', title='Buildings')]
).transform_filter(
    selection
).properties(
    title='When they were built', width=400
)

chart2 = usage_bars | decade_bars
chart2

alt.HConcatChart(...)

In [11]:
myJekyllSaveDir = '../assets/json/'
chart1.properties(width='container').save(myJekyllSaveDir + 'agency_sqft.json') 
chart2.save(myJekyllSaveDir + 'usage_decades.json')